Contents

- Test OpenReview API, on single name
  - Given name as string, try to get author id
  - Given author id, get status of all papers
- Get names from Airtable
- Get author ids from all names


In [66]:
import os
import openreview
from tqdm import tqdm
from datetime import datetime
import pandas as pd
from time import sleep

# Test OpenReview API

In [25]:
client = openreview.api.OpenReviewClient(
    baseurl='https://api2.openreview.net',
)

In [ ]:
name = "Lovkush Agarwal"
profiles = client.search_profiles(term=name)

In [27]:
print(len(profiles))
print(profiles[0])

1
{'active': True,
 'content': {'emails': ['****@gmail.com'],
             'emailsConfirmed': ['****@gmail.com'],
             'gender': 'Male',
             'gscholar': 'https://scholar.google.com/citations?view_op=list_works&hl=en&user=nm-XFVoAAAAJ',
             'history': [{'end': None,
                          'institution': {'country': 'GB',
                                          'domain': 'lovkush.com',
                                          'name': 'Independent'},
                          'position': 'Researcher',
                          'start': 2024},
                         {'end': 2020,
                          'institution': {'country': 'GB',
                                          'domain': 'le.ac.uk',
                                          'name': 'University of Leicester'},
                          'position': 'Instructor',
                          'start': 2017},
                         {'end': 2016,
                          'institution': {'countr

In [61]:
author_id = '~Lovkush_Agarwal1'
papers = client.get_all_notes(
    content={'authorids': author_id},
    details='replies'
)

In [62]:
print(papers[0])

{'cdate': 1724838331162,
 'content': {'TLDR': {'value': 'We study how language models might encode '
                               'paragraphs, and find newline tokens '
                               'activations do this to some extent.'},
             '_bibtex': {'value': '@inproceedings{\n'
                                  'pochinkov2024extracting,\n'
                                  'title={Extracting Paragraphs from {LLM} '
                                  'Token Activations},\n'
                                  'author={Nicky Pochinkov and Angelo Benoit '
                                  'and Lovkush Agarwal and Zainab Ali Majid '
                                  'and Lucile Ter-Minassian},\n'
                                  'booktitle={🍃 MINT: Foundation Model '
                                  'Interventions},\n'
                                  'year={2024},\n'
                                  'url={https://openreview.net/forum?id=4b675AHcqq}\n'
                   

In [ ]:
results = {}
for paper in papers:
    paper_info = {
        'title': paper.content['title']['value'],
        'date_creation': datetime.fromtimestamp(paper.cdate/1000).strftime('%Y-%m-%d'),
        # 'paperid': paper.id,
        'venueid': paper.content['venueid']['value'],
        'authorids': paper.content['authorids']['value'],
    }
    # Check for decision in replies
    decision = None
    for reply in paper.details['replies']:
        try:
            decision = reply['content']['decision']['value']
            break
        except:
            pass
    paper_info['decision'] = decision
    results[paper.id] = paper_info

for result in results:
    print(result)
    print(results[result])
    print()

4b675AHcqq
{'title': 'Extracting Paragraphs from LLM Token Activations', 'date_creation': '2024-08-28', 'venueid': 'NeurIPS.cc/2024/Workshop/MINT', 'authorids': ['~Nicky_Pochinkov1', '~Angelo_Benoit1', '~Lovkush_Agarwal1', '~Zainab_Ali_Majid1', '~Lucile_Ter-Minassian1'], 'decision': 'Accept'}



# Get names from Airtable

In [2]:
import os
import dotenv
from pyairtable import Api

dotenv.load_dotenv()
api = Api(os.environ['AIRTABLE_API_KEY'])

base_id = "appZq2f1sM0tW9kH7"
table_id = "tblX2K7X7sFF8Q8Be"
table = api.table(base_id, table_id)
table_data = table.all()


In [11]:
names = []
for row in table_data:
    name = row['fields']['Scholar Name']
    name = name.strip().lower() # Remove leading and trailing whitespace, make lower case
    if name == "":
        continue
    if name not in names:
        names.append(name)

# for name in names:
#     print(name)


# Get author ids from all names

In [17]:
# scholar info dict
# keys are names, value is dictionary with two keys: n_profiles and author_id if n_profiles is 1, otherwise None

client = openreview.api.OpenReviewClient(
    baseurl='https://api2.openreview.net',
)

scholar_info = {}

for name in tqdm(names):
    profiles = client.search_profiles(term=name)
    n_profiles = len(profiles)
    if n_profiles == 1:
        author_id = profiles[0].id
    else:
        author_id = None 
    scholar_info[name] = {
        'n_profiles': n_profiles,
        'author_id': author_id
    }

100%|██████████| 305/305 [00:56<00:00,  5.40it/s]


In [20]:
# count how many scholars have 1 profile
n_scholars_with_1_profile = sum(1 for scholar in scholar_info.values() if scholar['n_profiles'] == 1)
print(f"Number of scholars with 1 profile: {n_scholars_with_1_profile}")

# count how many scholars have 0 profiles
n_scholars_with_0_profiles = sum(1 for scholar in scholar_info.values() if scholar['n_profiles'] == 0)
print(f"Number of scholars with 0 profiles: {n_scholars_with_0_profiles}")

# count how many scholars have 2 or more profiles
n_scholars_with_2_or_more_profiles = sum(1 for scholar in scholar_info.values() if scholar['n_profiles'] >= 2)
print(f"Number of scholars with 2 or more profiles: {n_scholars_with_2_or_more_profiles}")

assert n_scholars_with_1_profile + n_scholars_with_0_profiles + n_scholars_with_2_or_more_profiles == len(names)

Number of scholars with 1 profile: 185
Number of scholars with 0 profiles: 74
Number of scholars with 2 or more profiles: 46


In [23]:
# for name, scholar in list(scholar_info.items())[:30]:
#     print(f"{name}, {scholar['n_profiles']}, {scholar['author_id']}")


In [24]:
# get list of author_ids where they are not None
author_ids = [scholar['author_id'] for scholar in scholar_info.values() if scholar['author_id'] is not None]
print(len(author_ids))



185


# get paper stats for all author_ids

In [67]:
all_papers = {}

for author_id in tqdm(author_ids):
    # wait to prevent rate limiting. seems to be limit of 60 requests per minute
    sleep(1)
    
    papers = client.get_all_notes(
        content={'authorids': author_id},
        details='replies'
    )

    for paper in papers:
        if paper.id in all_papers:
            continue
        paper_info = {
            'title': paper.content['title']['value'],
            'date_creation': datetime.fromtimestamp(paper.cdate/1000).strftime('%Y-%m-%d'),
            # 'paperid': paper.id,
            'venueid': paper.content['venueid']['value'],
            'authorids': paper.content['authorids']['value'],
        }
        # Check for decision in replies
        decision = None
        for reply in paper.details['replies']:
            try:
                decision = reply['content']['decision']['value']
                break
            except:
                pass
        paper_info['decision'] = decision
        all_papers[paper.id] = paper_info

 19%|█▉        | 35/185 [00:27<01:39,  1.50it/s]

Retrying request: GET /notes?content.authorids=~Thomas_Bush1&limit=1000&details=replies&after=DzGe40glxs&sort=id&count=false, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 61 requests, surpassing the limit of 60 requests. Please try again in 33 seconds (2025-10-02-3488902)', 'status': 429, 'details': {'limit': 60, 'remaining': 0, 'resetTime': '2025-10-02T13:35:40.017Z', 'used': 61, 'current': 61, 'reqId': '2025-10-02-3488902'}}, error: None


 37%|███▋      | 68/185 [01:28<01:44,  1.12it/s]

Retrying request: GET /notes?content.authorids=~Tim_Tian_Hua1&limit=1000&details=replies&sort=id&count=true, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 61 requests, surpassing the limit of 60 requests. Please try again in 33 seconds (2025-10-02-3493214)', 'status': 429, 'details': {'limit': 60, 'remaining': 0, 'resetTime': '2025-10-02T13:36:40.992Z', 'used': 61, 'current': 61, 'reqId': '2025-10-02-3493214'}}, error: None


 56%|█████▌    | 103/185 [02:30<01:00,  1.35it/s]

...

 76%|███████▌  | 140/185 [03:31<00:37,  1.20it/s]

Retrying request: GET /notes?content.authorids=~Andrew_Mackenzie1&limit=1000&details=replies&after=4VWnC5unAV&sort=id&count=false, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 61 requests, surpassing the limit of 60 requests. Please try again in 30 seconds (2025-10-02-3502530)', 'status': 429, 'details': {'limit': 60, 'remaining': 0, 'resetTime': '2025-10-02T13:38:41.702Z', 'used': 61, 'current': 61, 'reqId': '2025-10-02-3502530'}}, error: None


 94%|█████████▍| 174/185 [04:34<00:10,  1.01it/s]

Retrying request: GET /notes?content.authorids=~Oscar_Balcells_Obeso1&limit=1000&details=replies&after=EqF16oDVFf&sort=id&count=false, response: {'name': 'RateLimitError', 'message': 'Too many requests: You have made 61 requests, surpassing the limit of 60 requests. Please try again in 28 seconds (2025-10-02-3506893)', 'status': 429, 'details': {'limit': 60, 'remaining': 0, 'resetTime': '2025-10-02T13:39:41.975Z', 'used': 61, 'current': 61, 'reqId': '2025-10-02-3506893'}}, error: None


100%|██████████| 185/185 [05:12<00:00,  1.69s/it]


In [68]:
# convert dict to dataframe
df = pd.DataFrame(all_papers).T
df.reset_index(inplace=True)
df.rename(columns={'index': 'paperid'}, inplace=True)
df.head()

,paperid,title,date_creation,venueid,authorids,decision
0,qzsDKwGJyB,Measuring Progress in Dictionary Learning for ...,2024-05-30,ICML.cc/2024/Workshop/MI,"[~Adam_Karvonen1, ~Benjamin_Wright2, ~Can_Rage...",Accept (Oral)
1,qrU3yNfX0d,SAEBench: A Comprehensive Benchmark for Sparse...,2025-01-23,ICML.cc/2025/Conference,"[~Adam_Karvonen1, ~Can_Rager1, ~Johnny_Lin1, ~...",Accept (poster)
2,pZakRK1QHU,Linearly Structured World Representations in M...,2023-10-07,NeurIPS.cc/2023/Workshop/UniReps,"[~Michael_Ivanitskiy1, ~Alexander_F_Spies1, ~T...",Accept (Poster)
3,bKawydfGhb,An Adversarial Example for Direct Logit Attrib...,2024-05-29,ICML.cc/2024/Workshop/MI,"[~Jett_Janiak1, ~Can_Rager1, ~James_Dao1, ~Yeu...",Accept (Poster)
4,SCEdoGghcw,Measuring Progress in Dictionary Learning for ...,2024-05-15,NeurIPS.cc/2024/Conference,"[~Adam_Karvonen1, ~Benjamin_Wright2, ~Can_Rage...",Accept (poster)


In [69]:
df['paper_url'] = df['paperid'].apply(lambda x: f"https://openreview.net/forum?id={x}")
df.head()

,paperid,title,date_creation,venueid,authorids,decision,paper_url
0,qzsDKwGJyB,Measuring Progress in Dictionary Learning for ...,2024-05-30,ICML.cc/2024/Workshop/MI,"[~Adam_Karvonen1, ~Benjamin_Wright2, ~Can_Rage...",Accept (Oral),https://openreview.net/forum?id=qzsDKwGJyB
1,qrU3yNfX0d,SAEBench: A Comprehensive Benchmark for Sparse...,2025-01-23,ICML.cc/2025/Conference,"[~Adam_Karvonen1, ~Can_Rager1, ~Johnny_Lin1, ~...",Accept (poster),https://openreview.net/forum?id=qrU3yNfX0d
2,pZakRK1QHU,Linearly Structured World Representations in M...,2023-10-07,NeurIPS.cc/2023/Workshop/UniReps,"[~Michael_Ivanitskiy1, ~Alexander_F_Spies1, ~T...",Accept (Poster),https://openreview.net/forum?id=pZakRK1QHU
3,bKawydfGhb,An Adversarial Example for Direct Logit Attrib...,2024-05-29,ICML.cc/2024/Workshop/MI,"[~Jett_Janiak1, ~Can_Rager1, ~James_Dao1, ~Yeu...",Accept (Poster),https://openreview.net/forum?id=bKawydfGhb
4,SCEdoGghcw,Measuring Progress in Dictionary Learning for ...,2024-05-15,NeurIPS.cc/2024/Conference,"[~Adam_Karvonen1, ~Benjamin_Wright2, ~Can_Rage...",Accept (poster),https://openreview.net/forum?id=SCEdoGghcw


In [70]:
df['decision'].value_counts(dropna=False)

decision
None                         312
Accept (Poster)               54
Reject                        50
Accept (poster)               49
Accept                        24
Accept (Spotlight)            10
Accept (Oral)                  7
Accept-Main                    6
Accept (spotlight poster)      4
Accept (oral)                  4
Accept (spotlight)             2
Accept-Findings                1
Accept (WiPP)                  1
Name: count, dtype: int64

# Manually investigate papers with no decision

In [71]:
df[df['decision'].isna()].head()

,paperid,title,date_creation,venueid,authorids,decision,paper_url
8,GRzC65IqoO,Scaling Neuron Interpretability: A Decision Tr...,2025-08-23,NeurIPS.cc/2025/Workshop/MechInterp,"[~Srujananjali_Medicherla1, ~Aditya_Singh10, ~...",None,https://openreview.net/forum?id=GRzC65IqoO
10,c6ewu0sufB,Path Integral Optimiser: Global Optimisation v...,2024-09-28,NeurIPS.cc/2024/Workshop/OPT,"[~Max_McGuinness1, ~Eirik_Fladmark2, ~Francisc...",None,https://openreview.net/forum?id=c6ewu0sufB
14,pxaCoKZWVf,Narrow Finetuning Leaves Clearly Readable Trac...,2025-08-23,NeurIPS.cc/2025/Workshop/MechInterp,"[~Julian_Minder1, ~Clément_Dumas1, ~Stewart_Sl...",None,https://openreview.net/forum?id=pxaCoKZWVf
15,JGRtSAlQ3h,Robustly identifying concepts introduced durin...,2025-02-07,ICLR.cc/2025/Workshop/SLLM,"[~Julian_Minder1, ~Clément_Dumas1, ~Bilal_Chug...",None,https://openreview.net/forum?id=JGRtSAlQ3h
16,CdiCNA1Y2R,Overcoming Sparsity Artifacts in Crosscoders t...,2025-08-19,NeurIPS.cc/2025/Workshop/MechInterp,"[~Clément_Dumas1, ~Julian_Minder1, ~Caden_Juan...",None,https://openreview.net/forum?id=CdiCNA1Y2R


In [56]:
paper_id = "GRzC65IqoO"
paper = client.get_all_notes(id=paper_id, details='replies')
print(paper[0])

# when you go to website, https://openreview.net/forum?id=GRzC65IqoO, you do just see the paper without any replies or decisions...

{'cdate': 1755920754253,
 'content': {'TLDR': {'value': 'We train decision trees on OthelloGPT neuron '
                               'activations to identify thousands of logical '
                               'rule neurons.'},
             '_bibtex': {'value': '@inproceedings{\n'
                                  'medicherla2025scaling,\n'
                                  'title={Scaling Neuron Interpretability: A '
                                  'Decision Tree Approach to Othello{GPT}},\n'
                                  'author={Srujananjali Medicherla and Aditya '
                                  'Singh and Zihang Wen and Adam Karvonen and '
                                  'Can Rager},\n'
                                  'booktitle={Mechanistic Interpretability '
                                  'Workshop at NeurIPS 2025},\n'
                                  'year={2025},\n'
                                  'url={https://openreview.net/forum?id=GRzC65IqoO}\n'
     

In [72]:
paper_id = "c6ewu0sufB"

# Same with this. When you go to URL, just see the paper without any replies or decisions...